# Day 1 · Session 5 — RAG: Retrieval Augmented Generation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ksuresh/fdp-llm-agentic-ai/blob/main/day1-llm-foundations/05_rag_fundamentals.ipynb)

---
**Duration:** ~45 minutes  
**Goal:** Build a complete RAG pipeline from scratch — a chatbot that answers questions from your own documents.

**Use case:** A college knowledge assistant that answers questions about courses, faculty, and department information — using your own documents, not the LLM's training data.

---
## The RAG Architecture

```
INDEXING (done once):
  Your Documents
      ↓
  [Text Splitter]  →  Chunks (500 chars each)
      ↓
  [Embedding Model]  →  Vectors (1536 numbers each)
      ↓
  [Vector Store]  →  ChromaDB / FAISS (stored on disk)

RETRIEVAL (every query):
  User Question
      ↓
  [Embedding Model]  →  Query vector
      ↓
  [Vector Store]  →  Top-k most similar chunks
      ↓
  [LLM + Context]  →  Grounded Answer
```

In [ ]:
# ── Install all required packages ────────────────────────────
# Run this cell once. Takes about 2-3 minutes.
!pip install -q \
    langchain \
    langchain-openai \
    langchain-community \
    langchain-chroma \
    chromadb \
    faiss-cpu \
    pypdf \
    openai \
    python-dotenv

print("✅ All packages installed")

In [ ]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found. Please set it in your .env file.")

print(f"✅ OpenAI key loaded: {OPENAI_API_KEY[:8]}...")

---
## Part 1 — The Problem RAG Solves

First, let's prove the hallucination problem with domain-specific questions.

In [ ]:
client = OpenAI(api_key=OPENAI_API_KEY)

# Questions about a specific college — the LLM won't know the answers
questions = [
    "What is the fee structure for M.Tech CSE at PSG College of Technology?",
    "Who is the Head of the Department of AI & Data Science at our college?",
    "What are the prerequisites for the elective course 'Natural Language Processing' in 7th semester CSE?",
]

print("WITHOUT RAG — LLM answers from training data only:")
print("=" * 60)

for q in questions:
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": q}],
        max_tokens=150,
        temperature=0,
    )
    print(f"\n❓ {q}")
    print(f"🤖 {response.choices[0].message.content[:200]}")
    print("-" * 60)

print("\n⚠️  These answers are either made up or pulled from generic internet data.")
print("    RAG fixes this by grounding the LLM in YOUR documents.")

---
## Part 2 — Create a Sample Knowledge Base

We'll create a sample college document. In a real scenario, this would be your actual PDF files.

In [ ]:
# ── Sample college knowledge base ────────────────────────────
# In practice, replace this with real PDF files from your college

COLLEGE_DOCS = [
    {
        "source": "department_handbook.txt",
        "content": """
DEPARTMENT OF COMPUTER SCIENCE AND ENGINEERING
Academic Year 2025-2026 Handbook

HEAD OF DEPARTMENT: Dr. Rajesh Kumar, PhD (IIT Madras)
Email: hod.cse@college.edu | Phone: 0422-4344100

VISION: To be a centre of excellence in computing education and research.
MISSION: Produce graduates who are technically competent, ethically grounded, and globally employable.

PROGRAMMES OFFERED:
1. B.Tech Computer Science and Engineering (4 years, 240 credits)
   - Intake: 120 students per year
   - NAAC Grade: A++ (2024)
   
2. M.Tech Computer Science and Engineering (2 years, 80 credits)
   - Intake: 18 students per year
   - Specialisations: AI/ML, Cyber Security, Cloud Computing

3. PhD Computer Science and Engineering
   - Full-time and Part-time options available
   - Current scholars: 42

FEE STRUCTURE (2025-2026):
B.Tech CSE:
  - Tuition Fee: Rs. 90,000 per year
  - Hostel Fee: Rs. 55,000 per year (optional)
  - Total (Day Scholar): Rs. 1,05,000 per year (includes all fees)

M.Tech CSE:
  - Tuition Fee: Rs. 50,000 per semester
  - Total (2 years): Rs. 2,00,000 (GATE scholars: 50% scholarship)
  - GATE cutoff 2025: Score 650+ (General), 580+ (OBC)

PhD CSE:
  - Full-time: Rs. 25,000 per year (with CSIR/UGC NET: fully funded)
  - Part-time: Rs. 40,000 per year
"""
    },
    {
        "source": "btech_syllabus_sem7.txt",
        "content": """
B.TECH CSE — SEMESTER 7 SYLLABUS (2025-2026)

CORE SUBJECTS:

1. CS701 — Compiler Design (4 credits)
   Faculty: Dr. Priya Nair
   Prerequisites: Data Structures (CS301), Theory of Computation (CS501)
   Syllabus: Lexical Analysis, Parsing (LL, LR), Semantic Analysis, 
   Code Generation, Optimisation, LLVM basics
   Assessment: Internal 40% + End Semester 60%

2. CS702 — Distributed Systems (4 credits)
   Faculty: Dr. Suresh Babu
   Prerequisites: Operating Systems (CS401), Computer Networks (CS501)
   Syllabus: Consistency models, Consensus algorithms (Paxos, Raft),
   Distributed transactions, CAP theorem, MapReduce, Kubernetes
   Assessment: Internal 40% + End Semester 60%

3. CS703 — Information Security (3 credits)
   Faculty: Dr. Anitha Krishnan
   Prerequisites: Computer Networks (CS501)
   Syllabus: Cryptography (AES, RSA, ECC), PKI, Network security,
   Penetration testing, OWASP Top 10, Indian IT Act

ELECTIVE SUBJECTS (Choose 2 from):

4. CS7E1 — Natural Language Processing (3 credits)
   Faculty: Dr. Kavitha Selvaraj
   Prerequisites: Machine Learning (CS601), Python Programming (CS201)
   Syllabus: Text preprocessing, Word embeddings (Word2Vec, GloVe, BERT),
   Sequence models (LSTM, Transformer), Named Entity Recognition,
   Sentiment Analysis, Question Answering, Large Language Models
   Project: Build an NLP application using HuggingFace Transformers
   Assessment: Internal 30% + End Semester 40% + Project 30%

5. CS7E2 — Computer Vision (3 credits)
   Faculty: Dr. Ramesh Chandrasekaran
   Prerequisites: Machine Learning (CS601), Linear Algebra
   Syllabus: Image processing, CNN architectures (VGG, ResNet, EfficientNet),
   Object detection (YOLO, Faster R-CNN), Semantic segmentation,
   Transfer learning, Vision Transformers (ViT)

6. CS7E3 — Blockchain Technology (3 credits)
   Faculty: Dr. Manoj Kumar
   Prerequisites: Information Security (CS703) or equivalent
   Syllabus: Distributed ledger, Consensus (PoW, PoS), Smart contracts,
   Ethereum, Solidity, DeFi, NFT, Hyperledger Fabric

INTERNSHIP:
CS7I1 — Industry Internship (6 credits)
  Duration: Minimum 6 weeks
  Registration: Before Week 2 of Semester 7
  Report submission: Week 14
  Industry partners: Infosys, TCS, Wipro, HCL, local startups
"""
    },
    {
        "source": "faculty_research.txt",
        "content": """
FACULTY RESEARCH AREAS — CSE DEPARTMENT

Dr. Rajesh Kumar (HOD)
  Research: Deep Learning for Medical Imaging, Federated Learning
  Publications: 48 (SCI), h-index: 22
  Funded projects: DST-SERB (Rs. 42 lakhs), ICMR (Rs. 18 lakhs)
  PhD scholars: 8 (ongoing)

Dr. Kavitha Selvaraj (Associate Professor)
  Research: Natural Language Processing, Tamil NLP, Sentiment Analysis
  Publications: 31 (SCI/Scopus), h-index: 14
  Funded projects: TNSCST (Tamil Nadu language AI) Rs. 12 lakhs
  Notable: Developed TamilBERT — BERT model for Tamil language
  PhD scholars: 4 (ongoing)

Dr. Priya Nair (Associate Professor)
  Research: Compiler Optimisation, Program Analysis, WebAssembly
  Publications: 19 (SCI/Scopus), h-index: 9

Dr. Suresh Babu (Assistant Professor)
  Research: Edge Computing, IoT Security, Autonomous Systems
  Publications: 12 (SCI/Scopus), h-index: 6
  Current project: MeitY-funded smart agriculture IoT platform

Dr. Anitha Krishnan (Assistant Professor)
  Research: Cyber Security, Post-Quantum Cryptography, Zero-Trust Networks
  Publications: 15 (SCI/Scopus), h-index: 8
  Certifications: CEH, CISSP

RESEARCH LABS:
1. AI & Vision Lab (Room 301) — Dr. Rajesh Kumar
   Equipment: 4× NVIDIA A100 GPUs, Jetson clusters
2. NLP Lab (Room 302) — Dr. Kavitha Selvaraj
   Equipment: 2× RTX 4090 workstations, annotation tools
3. Cyber Security Lab (Room 303) — Dr. Anitha Krishnan
   Equipment: Penetration testing toolkit, Cisco network equipment

PLACEMENTS (2024-25 batch):
  Placement rate: 94.2%
  Highest package: Rs. 48 LPA (Google)
  Average package: Rs. 8.4 LPA
  Top recruiters: Google, Microsoft, Amazon, Zoho, Freshworks, TCS
  Dream companies (>15 LPA): 23% of placed students
"""
    },
    {
        "source": "exam_regulations.txt",
        "content": """
EXAMINATION REGULATIONS — B.TECH CSE (2025-26)

ASSESSMENT PATTERN:
Internal Assessment (40%):
  - Test 1: Week 6 (20 marks)
  - Test 2: Week 12 (20 marks)
  - Average of 2 tests = Internal marks
  - Assignment/Lab: 10 marks (separate from tests)

End Semester Examination (60%):
  - 3 hour written exam
  - 5 questions, choose 4 (each 15 marks)
  - Open book allowed for: CS7E1 (NLP), CS7E2 (Computer Vision)

GRADING:
  O  (Outstanding): 91-100 → 10 points
  A+ (Excellent):   81-90  → 9 points
  A  (Very Good):   71-80  → 8 points
  B+ (Good):        61-70  → 7 points
  B  (Average):     51-60  → 6 points
  RA (Re-Appear):   Below 50 → must reappear

PASS CRITERIA:
  - Minimum 50% in End Semester Exam independently
  - AND minimum 50% in total (Internal + End Sem combined)
  - Students must attend minimum 75% of classes (mandatory)
  - Below 75% attendance: Not permitted to write End Sem Exam

ARREAR POLICY:
  - Maximum 2 arrears to progress to next year
  - More than 2 arrears: Year detained
  - Arrear exams: April-May and November (separate from regular exams)

ACADEMIC CALENDAR 2025-26:
  Semester 1 (Odd): July 21, 2025 — November 28, 2025
  End Sem Exams 1:  December 1-20, 2025
  Semester 2 (Even): January 5, 2026 — May 15, 2026
  End Sem Exams 2:  May 18 — June 5, 2026
  Summer Internship: June 8 — July 18, 2026
"""
    }
]

# Save documents to files
import os
os.makedirs("college_docs", exist_ok=True)
for doc in COLLEGE_DOCS:
    with open(f"college_docs/{doc['source']}", "w") as f:
        f.write(doc['content'])

print(f"✅ Created {len(COLLEGE_DOCS)} knowledge base documents in college_docs/")
for doc in COLLEGE_DOCS:
    size = len(doc['content'])
    print(f"   📄 {doc['source']} — {size:,} characters")

---
## Part 3 — Step by Step: Understanding Each RAG Component

### Step 3.1 — Load Documents

In [ ]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from langchain.schema import Document

# Load all text files from the college_docs folder
loader = DirectoryLoader(
    "college_docs/",
    glob="*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"}
)
raw_docs = loader.load()

print(f"✅ Loaded {len(raw_docs)} documents")
for doc in raw_docs:
    print(f"   📄 {doc.metadata['source']} — {len(doc.page_content):,} chars")

# Peek at one document
print(f"\n--- Preview of first document ---")
print(raw_docs[0].page_content[:300] + "...")

### Step 3.2 — Split into Chunks

**Why chunk?** LLMs have context window limits. Large documents must be split into smaller pieces so we can retrieve only the relevant parts.

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# RecursiveCharacterTextSplitter tries to split on:
# paragraphs (\n\n) → sentences (\n) → words ( ) → characters
# This keeps semantically related text together.

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,       # max characters per chunk
    chunk_overlap=100,    # overlap between chunks (preserves context at boundaries)
    separators=["\n\n", "\n", ".", " ", ""],  # priority order for splitting
    length_function=len,
)

chunks = text_splitter.split_documents(raw_docs)

print(f"✅ Split into {len(chunks)} chunks")
print(f"   Average chunk size: {sum(len(c.page_content) for c in chunks) // len(chunks)} chars")
print(f"   Smallest chunk: {min(len(c.page_content) for c in chunks)} chars")
print(f"   Largest chunk:  {max(len(c.page_content) for c in chunks)} chars")

# Show a few chunks to understand the splitting
print(f"\n--- Sample chunks ---")
for i, chunk in enumerate(chunks[:3]):
    print(f"\nChunk {i+1} ({len(chunk.page_content)} chars):")
    print(f"Source: {chunk.metadata['source']}")
    print(chunk.page_content[:200])
    print("...")

### Step 3.3 — Create Embeddings

**What is an embedding?** Each chunk of text → a vector of 1536 numbers. Similar text = similar vectors = close in vector space.

In [ ]:
from langchain_openai import OpenAIEmbeddings

# text-embedding-3-small: 1536 dimensions, cheapest and excellent quality
# text-embedding-3-large: 3072 dimensions, more accurate but 5x the cost

embedding_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY
)

# Demo: see what an embedding looks like
sample_text = "What are the prerequisites for NLP elective?"
sample_vector = embedding_model.embed_query(sample_text)

print(f"✅ Embedding model ready: text-embedding-3-small")
print(f"   Input text: '{sample_text}'")
print(f"   Output vector: {len(sample_vector)} dimensions")
print(f"   First 8 values: {[round(v, 4) for v in sample_vector[:8]]}")
print(f"   (Each number encodes a facet of semantic meaning)")

In [ ]:
# ── Visualise similarity between texts ───────────────────────
import numpy as np

def cosine_similarity(v1, v2):
    """Measure how similar two vectors are. 1.0 = identical, 0.0 = unrelated."""
    v1, v2 = np.array(v1), np.array(v2)
    return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

texts = [
    "NLP elective prerequisites",
    "Natural Language Processing course requirements",
    "Fee structure for M.Tech CSE",
    "Cost of Masters programme in Computer Science",
    "Blockchain cryptocurrency Bitcoin",
]

vectors = [embedding_model.embed_query(t) for t in texts]

print("Semantic similarity scores (1.0 = identical, 0.0 = unrelated):")
print("-" * 65)
anchor = texts[0]
anchor_vec = vectors[0]
for i in range(1, len(texts)):
    sim = cosine_similarity(anchor_vec, vectors[i])
    bar = "█" * int(sim * 20)
    print(f"'{texts[i][:35]:<35}' → {sim:.3f} {bar}")

print(f"\nAnchor: '{anchor}'")
print("Notice: semantically similar phrases score high even with different words.")

### Step 3.4 — Store in Vector Database (ChromaDB)

In [ ]:
import shutil
from langchain_chroma import Chroma

# Clean up any previous vector store
if os.path.exists("./chroma_college_db"):
    shutil.rmtree("./chroma_college_db")

print("Building vector store... (this calls the OpenAI Embeddings API once)")

# This does 3 things in one call:
# 1. Embeds each chunk (calls OpenAI embedding API)
# 2. Stores vectors + original text in ChromaDB
# 3. Persists to disk (so you don't pay for re-embedding next time)
vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embedding_model,
    persist_directory="./chroma_college_db",
    collection_name="college_knowledge_base"
)

count = vectorstore._collection.count()
print(f"\n✅ Vector store created!")
print(f"   Vectors stored: {count}")
print(f"   Persisted to: ./chroma_college_db")
print(f"   Cost: ~{count * 0.00000002:.5f} USD = ₹{count * 0.00000002 * 83.5:.4f}")
print(f"   (Embedding is done ONCE. Query costs are tiny comparatively.)")

### Step 3.5 — Retrieve: Find Relevant Chunks

In [ ]:
# ── Bare retrieval: see what chunks get returned ──────────────
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}  # return top 3 most relevant chunks
)

test_query = "What are the prerequisites for the NLP elective?"
retrieved_docs = retriever.invoke(test_query)

print(f"Query: '{test_query}'")
print(f"Retrieved {len(retrieved_docs)} chunks:")
print("=" * 60)

for i, doc in enumerate(retrieved_docs):
    print(f"\nChunk {i+1} (from: {doc.metadata['source']}):")
    print(doc.page_content)
    print("-" * 60)

print("\n💡 These chunks will be injected into the LLM prompt as context.")

### Step 3.6 — Generate: LLM + Context = Grounded Answer

In [ ]:
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
from langchain.prompts import PromptTemplate

# Custom prompt that instructs the LLM to use ONLY the provided context
RAG_PROMPT = PromptTemplate(
    template="""You are a helpful academic assistant for students and faculty at a college.
Answer the question using ONLY the context provided below.
If the answer is not in the context, say: "I don't have this information in my knowledge base. 
Please contact the department office directly."

Context:
{context}

Question: {question}

Answer:""",
    input_variables=["context", "question"]
)

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0,            # 0 = deterministic, important for factual Q&A
    openai_api_key=OPENAI_API_KEY
)

# Build the RAG chain
rag_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",       # "stuff" = concatenate all retrieved docs into prompt
    retriever=retriever,
    chain_type_kwargs={"prompt": RAG_PROMPT},
    return_source_documents=True,  # so we can show which docs were used
)

print("✅ RAG chain ready")

In [ ]:
def ask_college_bot(question, show_sources=True):
    """Ask the college knowledge base a question."""
    result = rag_chain.invoke({"query": question})
    
    print(f"❓ Question: {question}")
    print(f"\n🤖 Answer: {result['result']}")
    
    if show_sources:
        sources = list(set(doc.metadata['source'] for doc in result['source_documents']))
        print(f"\n📚 Sources used: {', '.join(sources)}")
    print("-" * 70)

# Now test with the same questions — compare with the earlier without-RAG results!
print("WITH RAG — Grounded answers from your documents:")
print("=" * 70)

questions = [
    "What is the fee structure for M.Tech CSE?",
    "Who is the Head of the CSE Department?",
    "What are the prerequisites for the Natural Language Processing elective?",
    "What was the highest package in placements last year?",
    "When does Semester 1 end?",
]

for q in questions:
    ask_college_bot(q)
    print()

In [ ]:
# ── Test the "I don't know" behaviour ─────────────────────────
# The LLM should admit ignorance for things NOT in our documents

print("Testing out-of-scope questions (should admit ignorance):")
print("=" * 70)

out_of_scope = [
    "What is the recipe for biryani?",
    "Who won the IPL 2025 trophy?",
    "What is the admission process for next year's B.Tech?",  # not in our docs
]

for q in out_of_scope:
    ask_college_bot(q, show_sources=False)
    print()

---
## Part 4 — FAISS: Faster Alternative to ChromaDB

In [ ]:
from langchain_community.vectorstores import FAISS
import time

print("Building FAISS index...")
t0 = time.time()

# FAISS is an in-memory index — faster queries, can save/load to disk
faiss_store = FAISS.from_documents(
    documents=chunks,
    embedding=embedding_model
)

build_time = time.time() - t0

# Save to disk
faiss_store.save_local("./faiss_college_db")
print(f"✅ FAISS index built in {build_time:.1f}s")
print(f"   Saved to: ./faiss_college_db")

# Query speed comparison
t1 = time.time()
faiss_results = faiss_store.similarity_search("NLP prerequisites", k=3)
faiss_time = (time.time() - t1) * 1000

t2 = time.time()
chroma_results = vectorstore.similarity_search("NLP prerequisites", k=3)
chroma_time = (time.time() - t2) * 1000

print(f"\nQuery speed (smaller = faster):")
print(f"  FAISS:   {faiss_time:.1f} ms")
print(f"  ChromaDB:{chroma_time:.1f} ms")
print(f"\n💡 FAISS is faster for large collections.")
print(f"   ChromaDB is better when you need persistence + filtering.")

In [ ]:
# ── Load FAISS from disk (simulates a production scenario) ────
# In production: you build the index once, save it, then load on each server restart

loaded_faiss = FAISS.load_local(
    "./faiss_college_db",
    embeddings=embedding_model,
    allow_dangerous_deserialization=True  # required parameter in latest FAISS
)

print("✅ FAISS index loaded from disk successfully")

# Build RAG chain with FAISS instead of ChromaDB
faiss_rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=loaded_faiss.as_retriever(search_kwargs={"k": 3}),
    chain_type_kwargs={"prompt": RAG_PROMPT},
)

result = faiss_rag.invoke({"query": "What is the grading system?"})
print(f"\nFAISS RAG answer: {result['result']}")

---
## Part 5 — RAG with PDF Files (Real-world usage)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

def build_rag_from_pdf(pdf_path, collection_name="pdf_rag"):
    """
    Complete function: PDF file → working RAG chatbot
    5 lines of logic. Production-ready.
    """
    print(f"📄 Loading: {pdf_path}")
    
    # 1. Load PDF
    loader = PyPDFLoader(pdf_path)
    pages = loader.load()
    print(f"   Loaded {len(pages)} pages")
    
    # 2. Split into chunks
    splitter = RecursiveCharacterTextSplitter(chunk_size=600, chunk_overlap=100)
    chunks = splitter.split_documents(pages)
    print(f"   Split into {len(chunks)} chunks")
    
    # 3. Build vector store
    vs = FAISS.from_documents(chunks, embedding_model)
    print(f"   Indexed {len(chunks)} vectors")
    
    # 4. Build RAG chain
    chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vs.as_retriever(search_kwargs={"k": 4}),
        chain_type_kwargs={"prompt": RAG_PROMPT},
    )
    print(f"✅ RAG chatbot ready for: {pdf_path}")
    return chain


def chat_with_pdf(chain, question):
    result = chain.invoke({"query": question})
    return result['result']


# ── Usage instructions ────────────────────────────────────────
print("To use with your own PDF:")
print()
print('  # 1. Upload your PDF to the same folder as this notebook')
print('  # 2. Run:')
print('  chain = build_rag_from_pdf("your_document.pdf")')
print('  answer = chat_with_pdf(chain, "What is this document about?")')
print('  print(answer)')
print()
print("Try any PDF: textbook chapter, research paper, lab manual, annual report")

---
## Part 6 — Multi-Turn RAG Conversation

In [ ]:
from langchain.chains import ConversationalRetrievalChain
from langchain.memory import ConversationBufferMemory

# Memory stores the conversation history
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True,
    output_key="answer"
)

# ConversationalRetrievalChain: RAG + memory
# Automatically rewrites the question using chat history context
conv_rag = ConversationalRetrievalChain.from_llm(
    llm=llm,
    retriever=vectorstore.as_retriever(search_kwargs={"k": 3}),
    memory=memory,
    return_source_documents=False,
    verbose=False
)

# Simulate a student conversation
conversation = [
    "I'm interested in the NLP elective. What are the prerequisites?",
    "I've taken Machine Learning already. Who teaches this course?",
    "What kind of project will I do in that course?",
    "And how is the assessment split?",
]

print("Multi-turn conversation (the bot remembers context):")
print("=" * 60)

for turn in conversation:
    result = conv_rag.invoke({"question": turn})
    print(f"\n👤 Student: {turn}")
    print(f"🤖 Bot:     {result['answer']}")

---
## Part 7 — Chunking Strategy Comparison

In [ ]:
# Chunking strategy affects retrieval quality significantly
# Let's compare 3 strategies on the same question

from langchain.text_splitter import RecursiveCharacterTextSplitter

strategies = [
    {"name": "Small chunks",  "chunk_size": 200, "overlap": 20},
    {"name": "Medium chunks", "chunk_size": 500, "overlap": 100},
    {"name": "Large chunks",  "chunk_size": 1500, "overlap": 200},
]

test_query = "What are the prerequisites for NLP and who teaches it?"

print(f"Query: '{test_query}'")
print("Comparing retrieval quality across chunking strategies:")
print("=" * 65)

for strat in strategies:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=strat["chunk_size"],
        chunk_overlap=strat["overlap"]
    )
    strat_chunks = splitter.split_documents(raw_docs)
    strat_vs = FAISS.from_documents(strat_chunks, embedding_model)
    results = strat_vs.similarity_search(test_query, k=2)
    
    print(f"\n{strat['name']} (size={strat['chunk_size']}, {len(strat_chunks)} chunks):")
    for r in results:
        print(f"  → {r.page_content[:120].strip()}...")

print("\n💡 Medium chunks (400-600 chars) usually work best for Q&A.")
print("   Too small = loses context. Too large = retrieves irrelevant content.")

---
## 🎯 Lab Assignment

**File:** `lab/lab5_rag_chatbot.ipynb`

### Task 1 — Swap the knowledge base
Replace the `COLLEGE_DOCS` with documents from YOUR subject or institution. Good options:
- Your course syllabus
- Lab manual
- Department handbook
- Your own research paper

### Task 2 — PDF RAG
Use `build_rag_from_pdf()` with a real PDF. Ask 5 questions and check if the answers are accurate.

### Task 3 — Improve the prompt
Modify `RAG_PROMPT` to be more specific to your domain. For example:
- Add language instruction: "Answer in simple English, avoid jargon"
- Add citation instruction: "Always mention which section the answer comes from"
- Add format instruction: "If the answer involves numbers, present them in a table"

### Task 4 — Multi-document RAG
Add at least 3 different text/PDF documents. Ask questions that require combining information from multiple documents.

### 🌟 Bonus — Gradio UI
Build a simple web interface for your RAG chatbot:
```python
import gradio as gr

def respond(question, history):
    # Use conv_rag here
    result = conv_rag.invoke({"question": question})
    return result["answer"]

demo = gr.ChatInterface(respond, title="College Knowledge Bot")
demo.launch()
```